# كشكول التجارب: نماذج HuggingFace مع LangChain (المحاضرة 21)

أهلاً بيك! في الكشكول ده هنجرب إزاي نستخدم نماذج `HuggingFace` المجانية داخل `LangChain` مباشرة.

الميزة هنا إن `LangChain` بتقدملك واجهة موحدة فيها دالتين أساسيتين:
1. `embed_query("السؤال")`: بنستخدمها لما نيجي نسأل سؤال واحد.
2. `embed_documents(["جملة 1", "جملة 2"])`: بنستخدمها لما نيجي نخزن مجموعة مستندات مرة واحدة.

تعال نجربها ونشوف إزاي بتشتغل!


In [1]:
# 1. نستورد أداة التضمين من لانج تشين
from langchain_huggingface import HuggingFaceEmbeddings

# بنحدد الموديل وبنقوله خلي المتجهات مطبّعة (طولها = 1) عشان الحسابات تكون سريعة
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

print("تم تجهيز مسترجع HuggingFace بنجاح!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

تم تجهيز مسترجع HuggingFace بنجاح!


### 2. تجربة تضمين سؤال وتضمين مستندات
تعال نكتب سؤال ومجموعة وثائق ونشوف المتجهات:


In [2]:
# السؤال اللي المستخدم بيسأله
user_question = "How to build a search app using language models?"

# الوثائق الموجودة في قاعدة البيانات
my_documents = [
    "LangChain makes it easy to create RAG applications with vector databases.",
    "Python dictionaries store key-value pairs in memory.",
    "Baking sourdough bread requires flour, water, and active yeast starter."
]

# نحول السؤال لمتجه
q_vec = embeddings.embed_query(user_question)

# نحول الوثائق لمتجهات دفعة واحدة
doc_vecs = embeddings.embed_documents(my_documents)

print(f"طول متجه السؤال: {len(q_vec)} رقم")
print(f"عدد متجهات الوثائق: {len(doc_vecs)}")


طول متجه السؤال: 384 رقم
عدد متجهات الوثائق: 3


### 3. حساب درجة الشبه وترتيب المستندات
بما إن المتجهات مطبعة (`normalized`)، نقدر نحسب التشابه بضرب مصفوفات بسيط جداً وسريع:


In [3]:
import numpy as np

scores = np.dot(doc_vecs, q_vec)

print(f"السؤال: '{user_question}'\n")
print("النتائج مرتبة حسب القرب في المعنى:")
for idx in np.argsort(scores)[::-1]:
    print(f"  درجة {scores[idx]:.3f} -> {my_documents[idx]}")


السؤال: 'How to build a search app using language models?'

النتائج مرتبة حسب القرب في المعنى:
  درجة 0.332 -> LangChain makes it easy to create RAG applications with vector databases.
  درجة 0.103 -> Python dictionaries store key-value pairs in memory.
  درجة 0.029 -> Baking sourdough bread requires flour, water, and active yeast starter.


---
### مساحتك للتجربة الحرة
جرب تغير السؤال أو ضيف جمل جديدة وشوف الترتيب هيطلع إزاي!


In [4]:
# جرب هنا:
# test_q = "How do I make bread at home?"
# احسب الشبه وشوف مين اللي هيطلع الأول!
